# Layerwise ↔ FPM 对齐调查

- 模型:**Qwen3.6-35B-A3B**(数据是 3.6,不是 3.5),b300,vLLM 0.20.1
- 目的:**存记忆**。每节 = 问题 / 数据 / 结论
- 可复现:读本地 golden + backbone

In [ ]:
import pandas as pd, numpy as np, subprocess
from pathlib import Path
REPO = Path.cwd()
while not (REPO/"src/aiconfigurator").exists() and REPO != REPO.parent: REPO = REPO.parent
GOLDEN   = REPO/"fpm_golden_runs/fpm_upfront_qwen36_moe_full_once_20260613_201336/tp4_ep4_past4096/fpm_metrics_phase.csv"
BACKBONE = REPO/"runs/layerwise_qwen36_tp4ep4_cleanctx4/layerwise_native_tagged128.csv"
g = pd.read_csv(GOLDEN)
print("golden 步数:", g.phase.value_counts().to_dict())

## 大框架

- 每个 step = **backbone(非-MoE)** + **MoE overlay**
- 查了 4 个方向,各打在不同层:

| # | 怀疑 | 哪层 | 结论 |
|---|---|---|---|
| 1 | expert 不均衡/router | overlay | ❌ skew 自平均,power-law 够 |
| 2 | layerwise≠FPM 口径 | 测量 | ❌ 只占一小部分(ctx 计时,已修) |
| 3 | **MoE overlay 不准** | overlay | ✅ **已修**(decode 51→21%) |
| 4 | **small-prefill** | backbone | ⏸ **诊断清楚,暂缓** |

## golden 是啥

- **受控 sweep,不是生产 trace**
- 6945 步 = 70 ctx + 300 mixed + 6575 decode
- 前几十步故意全跑 pure prefill(无 decode)来干净测 context
- prefill 被切块:**大首块 + 小尾块**(528 对齐)

In [ ]:
# 前 10 步:重复的固定 prefill 模式(= 受控 sweep)
print(g[['phase','ctx_tokens','ctx_kv_tokens','decode_tokens']].head(10).to_string(index=False))
ctx = g[g.phase=='context']
print("\nctx_requests:", ctx.ctx_requests.value_counts().to_dict(), " (全 1 = 单请求)")
print("chunk 对齐: 528=1x528, 1056=2x528, 3696=7x528 ; 尾块=余数(496/357/400)")

## #3 — MoE overlay decode 重复计(✅ 已修)

- **问题**:decode MoE 过预测 ~10×
- **原因**:① diagnostics 把 `moe_inter` 写成 256(真 512)→ 命中错数据 ② fused kernel 已含 router/dispatch,AIC 又加一遍
- **修**:inter→512 + 删重复项(kernel-source-aware)
- **结果**:**decode MAPE 51→21%**;动吞吐 → 真赢点

In [ ]:
# BEFORE 数字来自 git-checkout 实验(记录);AFTER 可现场复现
print("BEFORE(fix前):  decode 50.9%   mixed 53.6%")
print("AFTER (fix后):  decode 19.3%   mixed 38.8%")
print("\n--- 复现 AFTER(当前代码)---")
cmd=["uv","run","python","collector/layerwise/diagnostics/compare_aic_layerwise_fpm_summary.py",
     "--layerwise",str(BACKBONE),"--moe-perf-file","collector/layerwise/wip/moe_perf.txt"]
out=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True).stdout
print("\n".join(l for l in out.splitlines() if "tp4_ep4" in l or l.startswith("case")))

## #4 — small-prefill backbone(⏸ 暂缓)

- **问题**:≤512 prefill 过预测 ~3×(AIC ~50ms vs golden ~16ms)
- **原因**:1-GPU 收集的 **launch-gap 假地板**(不是 comm / 拓扑 / capture)
- **golden 自己有台阶**:≤512=16ms(captured 快路径),>512=40ms(eager)
- AIC:>512 准、**只 ≤512 过预测**
- **暂缓**:bounded(只小尾块)、不动推荐、修它要 collector instrumentation

In [ ]:
# filter-off 暴露大首块:小块过预测,大块准 ; golden 512 有台阶
d = pd.DataFrame({'ctx_tokens':[128,400,496,528,3696],'ctx_kv':[0,3696,528,0,0],
                  'golden_ms':[15.9,16.4,16.2,40.6,43.7],'aic_ms':[37.0,51.5,51.8,39.3,57.4]})
d['err%'] = ((d.aic_ms/d.golden_ms-1)*100).round(0)
print(d.to_string(index=False))
print("\n≤512: golden ~16ms(captured) → AIC 过预测 2-3x")
print(">512: golden ~40ms(eager)    → AIC 准(-3% / +31%)")

## mixed step = **max**,不是 sum

- `mixed ≈ max(prefill_term, decode_term)`
- **decode_term**:~5ms,封顶 ~8ms(memory-bound)
- **prefill_term**:≤512 = ~16ms(captured 平地板),>512 = ~40ms+(eager)
- **有 prefill 就 prefill 赢**(16/40 > 8);纯 decode 才 ~5ms
- 想 decode 赢:要超重 decode(decode_term>16ms),这 workload 没有
- **AIC 错**:① sum 了(该 max)② ≤512 当 eager 算(该 captured 16ms)

In [ ]:
mix = g[g.phase=='mixed']; dec = g[g.phase=='decode']
print(f"decode 封顶: median {dec.latency_ms.median():.1f}  max {dec.latency_ms.max():.1f} ms")
print("\nmixed latency vs prefill chunk (decode load 都 ~31tok/~160k kv):")
for b,gg in mix.groupby(pd.cut(mix.ctx_tokens,[0,512,1024,2048,99999]),observed=True):
    print(f"  prefill {str(b):>13}: {gg.latency_ms.median():5.1f} ms")
print("  纯 decode(无 prefill):  ~5 ms")
print("\n→ 512 处跳 16→40ms ; 任何 prefill 都压过 decode")

## 总结

- ✅ **#3 已修**(decode 51→21%,动吞吐,已 commit + GPU 验证)
- ⏸ **#4 暂缓**(small-prefill,bounded,要 instrumentation)
- **建模规则**(记住):
  - mixed = **max(prefill, decode)**,不是 sum
  - prefill_term:≤512 = captured 平地板(~16ms),>512 = eager(随 size 涨)
  - decode_term:memory-bound,封顶低(~8ms)
- 详见同目录:`MOE_INVESTIGATION_SUMMARY.md` / `SMALL_PREFILL_FIX_DESIGN.md` / `DECODE_MOE_OVERLAY_VERDICT.md`